# Review Sentiment API — training in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValeriaZav26/review-sentiment-api/blob/main/train_in_colab.ipynb)

An alternative to the GitHub Actions workflow: train and explore the model interactively. Run the cells top to bottom.

## 1. Get the project

In [ ]:
!git clone -q https://github.com/ValeriaZav26/review-sentiment-api.git
%cd review-sentiment-api
!ls

## 2. Install dependencies
Versions are pinned so the model loads later on the server with the same scikit-learn.

If Colab shows **Restart session** after installation, click it and then run the `%cd` cell below before continuing.

In [ ]:
!pip install -q -r requirements-dev.txt

In [ ]:
%cd /content/review-sentiment-api
import sklearn; print('scikit-learn', sklearn.__version__)  # should be 1.5.2

## 3. Train and compare the models
Downloads Amazon Polarity, trains Naive Bayes and Logistic Regression (GridSearchCV), saves the best model and writes metrics into README.md.

In [ ]:
!python -m src.train

## 4. Look at the results

In [ ]:
import json
from IPython.display import Image, display
print(json.dumps(json.load(open('reports/metrics.json')), indent=2))
print(json.dumps(json.load(open('reports/top_words.json')), indent=2))
display(Image('reports/confusion_matrix.png'))

## 5. Run the tests (model + API)

In [ ]:
!python -m pytest -q

## 6. Try the API inside Colab
FastAPI's TestClient calls the app directly, no server needed.

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as client:
    print(client.get('/health').json())
    for text in ['Great quality, arrived quickly. Would buy again!',
                 'Stopped working after two days, very disappointed.']:
        print(text, '->', client.post('/predict', json={'text': text}).json())

## 7. Download the trained model and reports (optional)
The GitHub Actions workflow already commits the model, so this is only needed if you retrain with other settings.

In [ ]:
from google.colab import files
!zip -q -r trained_artifacts.zip models/model.joblib reports README.md
files.download('trained_artifacts.zip')